In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.retrieval import make_dense_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
ranker = make_dense_ranker()

rows = []
for cl in TARGET_CLAUSES:
    for i in dev_idx:
        c = contracts[i]
        top = ranker(c, cl)[0]
        rows.append({"clause": cl, "has_clause": len(c["gold"][cl]) > 0, "top_score": top["score"]})
scores = pd.DataFrame(rows)
scores.to_csv("../eval/abstention_scores_dev.csv", index=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
for cl in TARGET_CLAUSES:
    d = scores[scores.clause == cl]
    print(cl, "| present:", d.has_clause.sum(), "| absent:", (~d.has_clause).sum(),
          "| AUC:", round(roc_auc_score(d.has_clause, d.top_score), 3))

Governing Law | present: 306 | absent: 51 | AUC: 0.907
Renewal Term | present: 118 | absent: 239 | AUC: 0.816
Notice Period To Terminate Renewal | present: 71 | absent: 286 | AUC: 0.765
Termination For Convenience | present: 118 | absent: 239 | AUC: 0.629
Non-Compete | present: 75 | absent: 282 | AUC: 0.789
Cap On Liability | present: 190 | absent: 167 | AUC: 0.864
Uncapped Liability | present: 81 | absent: 276 | AUC: 0.82


In [3]:
def sweep(cl):
    d = scores[scores.clause == cl]
    out = []
    for t in np.arange(0.40, 0.80, 0.02):
        found = d.top_score >= t
        out.append({
            "threshold": round(t, 2),
            "found_when_present": round(found[d.has_clause].mean(), 2),
            "found_when_absent": round(found[~d.has_clause].mean(), 2),
        })
    return pd.DataFrame(out)

sweep("Non-Compete")

,threshold,found_when_present,found_when_absent
0,0.40,1.00,1.00
1,0.42,1.00,1.00
2,0.44,1.00,1.00
3,0.46,1.00,1.00
4,0.48,1.00,1.00
5,0.50,1.00,0.99
6,0.52,1.00,0.96
7,0.54,1.00,0.94
8,0.56,1.00,0.90
9,0.58,1.00,0.84
